# Synthetic dataset generator — bank customer complaints

This notebook builds the CSV file used in **Demo 1 — Customer complaint classification**.
It is a *trainer* notebook: participants only see the resulting `bank_complaints.csv`.

The dataset is fully synthetic, but it is designed to contain the phenomena we want participants to
discover during exploratory data analysis (EDA) and modelling:

| Phenomenon | Where it is injected | How participants can detect it |
|---|---|---|
| Seasonality and an operational incident (mobile-app outage, February 2026) | complaint dates | line plot of monthly volumes |
| A "severity" structure: bigger amounts, longer texts, more negative sentiment, more contacts move together | numeric features (latent factor model) | correlation matrix, scatter matrix |
| The outcome depends on severity, channel, product and segment | target `outcome` | crosstabs, box plots by outcome, feature importance |
| A post-closure column that leaks the outcome | `compensation_eur` | "too good to be true" accuracy |
| Missing values: MCAR, MAR, MNAR, plus a missing value written as text | `customer_age`, `sentiment_score`/`text_length_words`, `disputed_amount`, `customer_segment` | missing rates by category |
| Univariate outliers | `disputed_amount`, `customer_age`, `text_length_words`, `days_since_event` | box plot, z-score |
| Bivariate outliers (each value is normal, the pair is not) | `customer_age` vs `tenure_years`, `tenure_years` vs `products_held` | scatter plot, scatter matrix |
| Multivariate outliers (every pair looks normal, the combination is not) | 8 numeric features | Mahalanobis distance only |
| Duplicated rows and inconsistent category labels | whole rows, `channel` | `duplicated()`, `value_counts()` |
| Short complaint texts (excerpts) consistent with product, tone and amount | `complaint_excerpt` | the LLM step of the demo; the true amount as a forensic clue |

Two files are written at the end:

- `bank_complaints.csv` — the dataset given to participants;
- `bank_complaints_answer_key.csv` — the list of injected anomalies (for the trainer only).

Everything is seeded, so re-running the notebook reproduces exactly the same files.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2

rng = np.random.default_rng(2026)   # one generator for everything -> fully reproducible

N_BASE = 2800    # "normal" complaints
N_SPIKE = 200    # extra complaints caused by the mobile-app incident (February 2026)

## 1. Complaint dates: seasonality, a mild trend and weekday effects

Complaints are spread over 18 months (January 2025 – June 2026). Monthly weights add a light seasonality
(more complaints in January, fewer in August) and a small upward trend in 2026. Most complaints filed at the
weekend are moved to the following Monday, as happens in a real complaint-management system that
registers complaints on working days.

In [ ]:
months = pd.period_range("2025-01", "2026-06", freq="M")

# relative weight of each month (18 values): seasonality + slight upward trend in 2026
month_weight = np.array([1.15, 1.05, 1.00, 0.95, 0.95, 0.90, 0.85, 0.70, 1.00, 1.05, 1.05, 0.90,   # 2025
                         1.20, 1.10, 1.05, 1.00, 1.00, 0.95])                                       # 2026
month_weight = month_weight / month_weight.sum()

month_idx = rng.choice(len(months), size=N_BASE, p=month_weight)
m = months[month_idx]
dates = pd.DatetimeIndex(m.start_time.values + pd.to_timedelta(rng.integers(0, m.days_in_month), unit="D"))

# 75% of weekend complaints are registered on the next Monday
weekday = dates.dayofweek.values
to_monday = (weekday >= 5) & (rng.random(N_BASE) < 0.75)
dates = dates + pd.to_timedelta(np.where(to_monday, 7 - weekday, 0), unit="D")

## 2. Customer and product attributes

Segment and product are drawn with realistic frequencies. They will later shift the disputed amount
(private-banking and investment complaints involve more money) and the outcome.

In [ ]:
segments = np.array(["Retail", "Affluent", "SME", "Private"])
segment_p = [0.65, 0.20, 0.10, 0.05]

products = np.array(["Current account", "Credit card", "Personal loan", "Mortgage", "Investments", "Insurance"])
product_p = [0.30, 0.25, 0.12, 0.13, 0.12, 0.08]

segment = rng.choice(segments, N_BASE, p=segment_p)
product = rng.choice(products, N_BASE, p=product_p)

## 3. Numeric features from a latent factor model

The eight numeric features are generated from two hidden ("latent") factors:

- a **severity factor**: the more serious the problem, the larger the disputed amount, the longer the
  complaint text, the more negative the sentiment, the more contacts before the formal complaint, the older
  the disputed event and the more previous complaints;
- a **customer maturity factor**: age and tenure grow together.

Each feature = loading × factor + independent noise. Loadings of 0.75 give pairwise correlations of about
0.56 inside the severity block. This is what makes the correlation matrix and the scatter matrix interesting
in the demo, and it is also the structure that the Mahalanobis distance exploits later.

Columns of the latent matrix `Z`, in order: age, tenure, log(amount), text length, sentiment, days since
event, contacts before complaint, prior complaints.

In [ ]:
# loadings on (severity, maturity); sentiment loads negatively on severity
loadings = np.array([[0.00, 0.806],    # age
                     [0.00, 0.806],    # tenure
                     [0.75, 0.15 ],    # log(disputed amount)
                     [0.75, 0.00 ],    # text length
                     [-0.75, 0.00],    # sentiment
                     [0.75, 0.00 ],    # days since event
                     [0.75, 0.00 ],    # contacts before complaint
                     [0.75, 0.00 ]])   # prior complaints
uniqueness = np.sqrt(1 - (loadings ** 2).sum(axis=1))   # noise std so that each latent column has variance 1

factors = rng.standard_normal((N_BASE, 2))      # column 0 = severity, column 1 = maturity
noise = rng.standard_normal((N_BASE, 8))
Z = factors @ loadings.T + noise * uniqueness
severity = factors[:, 0]                        # kept aside: it drives the outcome

# --- transform latent columns into business quantities ---
seg_age_shift = pd.Series(segment).map({"Retail": 0, "Affluent": 5, "SME": 3, "Private": 8}).values
customer_age = np.clip(np.round(46 + 13 * Z[:, 0] + seg_age_shift), 18, 92)

tenure_years = np.clip(np.round(9 + 7 * Z[:, 1], 1), 0.1, None)
tenure_years = np.minimum(tenure_years, customer_age - 18)      # nobody is a customer before age 18

seg_mult = pd.Series(segment).map({"Retail": 1.0, "Affluent": 1.6, "SME": 2.2, "Private": 2.6}).values
prod_mult = pd.Series(product).map({"Current account": 0.6, "Credit card": 0.7, "Personal loan": 1.2,
                                    "Mortgage": 2.2, "Investments": 2.6, "Insurance": 1.0}).values
disputed_amount = np.round(np.exp(6.0 + 1.1 * Z[:, 2] + np.log(seg_mult) + np.log(prod_mult)), 2)   # log-normal

text_length_words = np.clip(np.round(170 + 55 * Z[:, 3]), 20, None)
sentiment_score = np.round(np.clip(-0.30 + 0.30 * Z[:, 4], -1, 1), 2)

prod_days_shift = pd.Series(product).map({"Mortgage": 15, "Investments": 12}).fillna(0).values
days_since_event = np.clip(np.round(24 + 12 * Z[:, 5] + prod_days_shift), 0, None)

contacts_before_complaint = np.clip(np.round(2.5 + 1.4 * Z[:, 6]), 0, None)
prior_complaints_12m = np.clip(np.round(0.9 + 0.9 * Z[:, 7]), 0, None)

seg_products_shift = pd.Series(segment).map({"Retail": 0, "Affluent": 1, "SME": 0, "Private": 2}).values
products_held = np.clip(np.round(1.2 + 0.28 * tenure_years + 0.7 * rng.standard_normal(N_BASE) + seg_products_shift), 1, 8)

## 4. Channel depends on age

Younger customers complain through the mobile app and social media, older customers through the branch and
the phone. Channel probabilities are therefore chosen per age band.

In [ ]:
channels = np.array(["Phone", "Email", "Web form", "Mobile app", "Branch", "Social media"])
p_young  = [0.20, 0.18, 0.15, 0.28, 0.06, 0.13]   # age < 35
p_middle = [0.30, 0.25, 0.15, 0.13, 0.12, 0.05]   # 35 - 60
p_senior = [0.35, 0.20, 0.10, 0.05, 0.28, 0.02]   # > 60

P = np.where((customer_age < 35)[:, None], [p_young],
             np.where((customer_age <= 60)[:, None], [p_middle], [p_senior]))
channel = channels[(rng.random(N_BASE)[:, None] > P.cumsum(axis=1)).sum(axis=1)]

## 5. The target: complaint outcome

A hidden score combines severity, channel, product, segment and repeat-complainer behaviour, plus noise.
The score is cut at its 45th and 85th percentiles, giving three classes with a realistic imbalance:
45% resolved at first contact, 40% resolved after investigation, 15% escalated to the ombudsman.

Social-media complaints and investment/mortgage complaints escalate more often; mobile-app and branch
complaints less often. These are the "business insights" the demo should surface.

In [ ]:
channel_effect = pd.Series(channel).map({"Phone": 0.0, "Email": 0.1, "Web form": 0.1, "Mobile app": -0.35,
                                         "Branch": -0.25, "Social media": 1.0}).values
product_effect = pd.Series(product).map({"Current account": -0.3, "Credit card": -0.2, "Personal loan": 0.1,
                                         "Mortgage": 0.5, "Investments": 0.7, "Insurance": 0.2}).values
segment_effect = pd.Series(segment).map({"Retail": 0.0, "Affluent": 0.1, "SME": 0.25, "Private": 0.3}).values

score = (1.2 * severity + channel_effect + product_effect + segment_effect
         + 0.25 * (prior_complaints_12m >= 2) + 0.9 * rng.standard_normal(N_BASE))

t1, t2 = np.quantile(score, [0.45, 0.85])
outcome = np.where(score < t1, "Resolved at first contact",
          np.where(score < t2, "Resolved after investigation", "Escalated to ombudsman"))

base = pd.DataFrame({
    "complaint_date": dates, "product": product, "channel": channel, "customer_segment": segment,
    "customer_age": customer_age, "tenure_years": tenure_years, "products_held": products_held,
    "disputed_amount": disputed_amount, "prior_complaints_12m": prior_complaints_12m,
    "contacts_before_complaint": contacts_before_complaint, "days_since_event": days_since_event,
    "text_length_words": text_length_words, "sentiment_score": sentiment_score, "outcome": outcome})

## 6. The incident spike: mobile-app outage, February 2026

200 extra complaints arrive between 3 and 14 February 2026, all through the mobile app, all about the
current account, with small amounts and mostly resolved at first contact. In the demo this appears as a
clear spike in the monthly line plot, and it is a good opportunity to talk about operational incidents,
early-warning systems and data drift.

In [ ]:
spike_dates = pd.DatetimeIndex(pd.to_datetime("2026-02-03") + pd.to_timedelta(rng.integers(0, 12, N_SPIKE), unit="D"))
weekday = spike_dates.dayofweek.values
spike_dates = spike_dates + pd.to_timedelta(np.where(weekday >= 5, 7 - weekday, 0), unit="D")

zs = rng.standard_normal((N_SPIKE, 8))
spike_age = np.clip(np.round(34 + 10 * zs[:, 0]), 18, 80)
spike_tenure = np.minimum(np.clip(np.round(5 + 4 * zs[:, 1], 1), 0.1, None), spike_age - 18)

spike = pd.DataFrame({
    "complaint_date": spike_dates, "product": "Current account", "channel": "Mobile app",
    "customer_segment": rng.choice(segments, N_SPIKE, p=[0.80, 0.15, 0.05, 0.00]),
    "customer_age": spike_age, "tenure_years": spike_tenure,
    "products_held": np.clip(np.round(1.2 + 0.28 * spike_tenure + 0.7 * zs[:, 2]), 1, 8),
    "disputed_amount": np.round(np.exp(3.8 + 0.5 * zs[:, 3]), 2),                 # ~ 45 EUR
    "prior_complaints_12m": np.clip(np.round(0.3 + 0.6 * zs[:, 4]), 0, None),
    "contacts_before_complaint": np.clip(np.round(1 + zs[:, 5]), 0, None),
    "days_since_event": rng.integers(0, 6, N_SPIKE),
    "text_length_words": np.clip(np.round(90 + 30 * zs[:, 6]), 20, None),
    "sentiment_score": np.round(np.clip(-0.45 + 0.25 * zs[:, 7], -1, 1), 2),
    "outcome": rng.choice(["Resolved at first contact", "Resolved after investigation", "Escalated to ombudsman"],
                          N_SPIKE, p=[0.85, 0.13, 0.02])})

df = pd.concat([base, spike], ignore_index=True)
df = df.sort_values("complaint_date", kind="stable").reset_index(drop=True)   # a system export is ordered by date
df.insert(0, "complaint_id", [f"CMP-{i + 1:06d}" for i in range(len(df))])
print(df.shape)
df["outcome"].value_counts(normalize=True).round(3)

## 7. A post-closure column: `compensation_eur` (the leakage trap)

The compensation paid to the customer is known only when the complaint is closed, and it is almost a
mirror of the outcome: nothing for most first-contact resolutions, always something for escalations.
It is a legitimate column in a historical extract, but it must **not** be used by a model that runs at intake.
The demo uses it to show what "data leakage" looks like.

In [ ]:
n = len(df)
u = rng.random(n)          # "is any compensation paid?"
r = rng.random(n)          # "how much?"
amt = df["disputed_amount"].values
oc = df["outcome"].values

df["compensation_eur"] = np.where(
    oc == "Resolved at first contact", np.where(u < 0.97, 0.0, np.round(5 + 45 * r, 2)),
    np.where(oc == "Resolved after investigation", np.where(u < 0.10, 0.0, np.round(amt * (0.1 + 0.4 * r), 2)),
             np.round(amt * (0.3 + 0.9 * r) + 200, 2)))       # escalated: always paid, plus ombudsman fee

df.groupby("outcome")["compensation_eur"].agg(["mean", "median", lambda s: (s == 0).mean()]).round(2)

## 8. Messy layer, part 1: missing values

Three mechanisms, one per column, so that participants can *see* the difference:

- **MCAR** (missing completely at random): `customer_age` is blank in 4% of rows, regardless of anything else.
- **MAR** (missing at random): `sentiment_score` and `text_length_words` come from the complaint text. Branch
  and phone complaints often have no written text, so the blank depends on an *observed* column (`channel`).
- **MNAR** (missing not at random): `disputed_amount` is blank more often when the amount is *large*
  (large claims are managed in a separate legal workflow and the field stays empty). The blank depends on
  the *unobserved* value itself.

A fourth case is a missing value written as text: `customer_segment = "unknown"`, which `pandas` does not
recognise as missing.

Missing values are injected before the outliers, so that the outlier rows keep complete values.

In [ ]:
answer_key = []      # (complaint_id, anomaly description) -> trainer's answer key
used = set()         # rows already used for an injected anomaly

# candidate rows for injections: not part of the incident spike
is_base = ~((df["channel"] == "Mobile app") & (df["product"] == "Current account"))

def pick_rows(k, condition=None):
    """Pick k unused base rows (optionally satisfying a boolean condition)."""
    mask = is_base & ~df.index.isin(list(used))
    if condition is not None:
        mask &= condition
    idx = rng.choice(df.index[mask], k, replace=False)
    used.update(idx)
    return idx

# MCAR: 4% of ages, completely at random
mcar = rng.random(n) < 0.04
df.loc[mcar, "customer_age"] = np.nan

# MAR: text-derived features missing mostly for branch and phone complaints
p_mar = np.where(df["channel"].isin(["Branch", "Phone"]), 0.55, 0.02)
mar = rng.random(n) < p_mar
df.loc[mar, ["sentiment_score", "text_length_words"]] = np.nan

# MNAR: probability of a blank amount grows with the amount itself (logistic curve centred at 3,000 EUR)
log_amt = np.log(df["disputed_amount"])
p_mnar = 0.02 + 0.45 / (1 + np.exp(-(log_amt - np.log(3000)) / 0.6))
mnar = rng.random(n) < p_mnar
amount_before_mnar = df["disputed_amount"].copy()      # kept for the complaint texts (section 11)
df.loc[mnar, "disputed_amount"] = np.nan

# missing value written as text
unk = pick_rows(40)
df.loc[unk, "customer_segment"] = "unknown"
answer_key += [(c, "missing value written as text: customer_segment = 'unknown'") for c in df.loc[unk, "complaint_id"]]

print(f"MCAR age: {mcar.sum()} rows, MAR sentiment/text: {mar.sum()} rows, MNAR amount: {mnar.sum()} rows, 'unknown' segment: {len(unk)} rows")

## 9. Messy layer, part 2: univariate and bivariate outliers

**Univariate** outliers stand out on a single column (box plot, z-score):

- five disputed amounts multiplied by 100 (cents keyed as euros — a classic data-entry error);
- three impossible ages (127, 134, 119);
- two enormous complaint texts (a whole contract pasted in);
- one event that happened four years before the complaint.

**Bivariate** outliers look normal column by column, and are visible only when two columns are plotted together:

- six customers whose tenure is longer than their adult life (`tenure_years > customer_age - 18`);
- four customers with a few months of tenure and 7–8 products.

In [ ]:
# --- univariate ---
i = pick_rows(5, (df["customer_segment"] == "Retail") & df["disputed_amount"].between(200, 1500) & df["sentiment_score"].notna())
df.loc[i, "disputed_amount"] = np.round(df.loc[i, "disputed_amount"] * 100, 2)
answer_key += [(c, "univariate outlier: disputed_amount x100 (cents keyed as euros)") for c in df.loc[i, "complaint_id"]]

i = pick_rows(3)
df.loc[i, "customer_age"] = [127, 134, 119]
answer_key += [(c, "univariate outlier: impossible customer_age") for c in df.loc[i, "complaint_id"]]

i = pick_rows(2)
df.loc[i, "text_length_words"] = [9800, 12400]
answer_key += [(c, "univariate outlier: extreme text_length_words (document pasted in)") for c in df.loc[i, "complaint_id"]]

i = pick_rows(1)
df.loc[i, "days_since_event"] = 1450
answer_key += [(c, "univariate outlier: extreme days_since_event") for c in df.loc[i, "complaint_id"]]

# --- bivariate ---
i = pick_rows(6)
df.loc[i, "customer_age"] = [23, 26, 29, 24, 31, 27]
df.loc[i, "tenure_years"] = [21, 24, 26, 20, 25, 22]
answer_key += [(c, "bivariate outlier: tenure_years > customer_age - 18") for c in df.loc[i, "complaint_id"]]

i = pick_rows(4, df["customer_age"] >= 25)
df.loc[i, "tenure_years"] = [0.3, 0.5, 0.4, 0.6]
df.loc[i, "products_held"] = [8, 7, 8, 8]
answer_key += [(c, "bivariate outlier: months of tenure with 7-8 products") for c in df.loc[i, "complaint_id"]]

## 10. Messy layer, part 3: multivariate outliers (Mahalanobis only)

We want eight rows that:

1. are **not** univariate outliers (every value is within about ±2.2 standard deviations),
2. are **not** visible in any two-dimensional scatter plot (every pairwise Mahalanobis distance² stays below
   11, well inside the cloud of 3,000 points, whose natural maximum is around 16),
3. but have a large **eight-dimensional** Mahalanobis distance² (above the 99.9% chi-square threshold).

Such points exist because the eight numeric features are correlated: a point can be slightly off in several
directions at once, each deviation harmless on its own, but implausible in combination.

The search below uses exactly the same numeric columns that the demo notebook uses for the Mahalanobis
distance (with `log1p` applied to the amount). It draws 30,000 random directions in the "whitened" feature
space, computes for each direction how far we can push a point before it becomes visible in one dimension or
in one pair, and keeps the directions that allow the largest overall distance. Points are then rounded to
realistic values (integer ages and counts, cents on the amount).

In [ ]:
numeric_cols = ["customer_age", "tenure_years", "disputed_amount", "text_length_words", "sentiment_score",
                "days_since_event", "contacts_before_complaint", "prior_complaints_12m"]

# reference sample: rows without injected anomalies and without gross errors, complete cases only
reference = df[~df["complaint_id"].isin([c for c, _ in answer_key])]
reference = reference[(reference["customer_age"] <= 100) & (reference["tenure_years"] <= reference["customer_age"] - 18)]
X0 = reference[numeric_cols].copy()
X0["disputed_amount"] = np.log1p(X0["disputed_amount"])
X0 = X0.dropna()

mu = X0.mean().values
sd = X0.std().values
S = np.cov(X0.values, rowvar=False)
Lchol = np.linalg.cholesky(S)                         # S = L L^T ; a step L @ w has Mahalanobis length |w|
pairs = [(a, b) for a in range(8) for b in range(a + 1, 8)]
pair_inv = {p: np.linalg.inv(S[np.ix_(p, p)]) for p in pairs}

candidates = []
W = rng.standard_normal((30000, 8))
W /= np.linalg.norm(W, axis=1, keepdims=True)          # random unit directions in whitened space
for w in W:
    v = Lchol @ w                                       # the same direction in raw feature space (unit Mahalanobis length)
    z_per_unit = np.abs(v) / sd                          # univariate z-score gained per unit of distance
    pair_d2_per_unit = max(v[list(p)] @ pair_inv[p] @ v[list(p)] for p in pairs)
    # largest distance^2 that keeps every |z| < 2.2 and every pairwise D2 < 11
    max_d2 = min(11.0 / pair_d2_per_unit, (2.2 / z_per_unit.max()) ** 2)
    candidates.append((max_d2, w))
candidates.sort(key=lambda t: -t[0])

# rounding to realistic values (integer ages and counts, cents) moves a point a little, so we round first
# and accept a candidate only if the rounded point still passes the three checks
inv_S = np.linalg.inv(S)
threshold_8d = chi2.ppf(0.999, df=8)
chosen = []                                   # accepted rounded points
for d2, w in candidates:
    if any(abs(w @ c_w) > 0.8 for _, c_w in chosen):        # keep directions different from each other
        continue
    x = mu + np.sqrt(d2) * (Lchol @ w)                       # push the point to the reachable distance
    x_round = np.array([round(x[0]), round(min(max(x[1], 0.1), round(x[0]) - 18), 1), np.log1p(round(float(np.expm1(x[2])), 2)),
                        max(round(x[3]), 20), round(float(np.clip(x[4], -1, 1)), 2), max(round(x[5]), 0),
                        max(round(x[6]), 0), max(round(x[7]), 0)])
    diff = x_round - mu
    d2_round = diff @ inv_S @ diff
    z_round = np.abs(diff) / sd
    pair_round = max(diff[list(p)] @ pair_inv[p] @ diff[list(p)] for p in pairs)
    if d2_round > threshold_8d + 2 and z_round.max() < 2.3 and pair_round < 11.5:
        chosen.append((x_round, w))
    if len(chosen) == 8:
        break

print("Accepted points:", len(chosen))
print("99.9% chi-square threshold with 8 features:", round(threshold_8d, 1))

In [ ]:
i = pick_rows(8)
for row, (x, w) in zip(i, chosen):
    df.loc[row, "customer_age"] = int(x[0])
    df.loc[row, "tenure_years"] = float(x[1])
    df.loc[row, "disputed_amount"] = round(float(np.expm1(x[2])), 2)
    df.loc[row, "text_length_words"] = int(x[3])
    df.loc[row, "sentiment_score"] = float(x[4])
    df.loc[row, "days_since_event"] = int(x[5])
    df.loc[row, "contacts_before_complaint"] = int(x[6])
    df.loc[row, "prior_complaints_12m"] = int(x[7])
    # neutral categorical values, so that the anomaly lives only in the numeric combination
    df.loc[row, ["product", "channel", "customer_segment"]] = ["Current account", "Email", "Retail"]
answer_key += [(c, "multivariate outlier: detectable only with the Mahalanobis distance") for c in df.loc[i, "complaint_id"]]
df.loc[i, ["complaint_id"] + numeric_cols]

## 11. Complaint texts: the opening lines of each written complaint

The demo notebook shows the step that turns text into numbers by calling an LLM on a few complaints. For
that we need some text. We build, for every complaint, a short **excerpt** (the opening lines, 25–60 words)
from sentence pools: an opening sentence that depends on the product, a tone sentence that depends on the
sentiment score, and optional sentences that reflect the number of contacts, the previous complaints and the
age of the event. The mobile-app incident gets its own openings.

Two details make the texts useful in the demo:

- the amount mentioned in the text is the *true* amount, also when the field has been blanked (MNAR) or
  multiplied by 100 (the keying error): the text is the forensic clue;
- complaints with no written text (the MAR blanks of section 8) get no excerpt, so the three text-derived
  columns are missing together.

The full length of the complaint stays in `text_length_words`; the excerpt is only its opening.

In [ ]:
opening = {
    "Current account": [
        "An unauthorised direct debit of {amount} euros was taken from my current account on the {day}th.",
        "My current account was blocked for three days without any notice, and a transfer of {amount} euros bounced.",
        "I was charged {amount} euros in fees that were never mentioned when I opened the account.",
        "A transfer of {amount} euros I sent on the {day}th never reached the beneficiary and nobody can tell me where it is."],
    "Credit card": [
        "My card was charged twice for the same purchase of {amount} euros.",
        "There is a transaction of {amount} euros on my card statement that I never made.",
        "My card was blocked while I was abroad and I had to pay {amount} euros in cash with no explanation.",
        "The cash-advance fee of {amount} euros on my card is not the one stated in the terms."],
    "Personal loan": [
        "The instalment taken for my loan is {amount} euros higher than the amount agreed in the contract.",
        "I asked to repay my loan early and the fee applied, {amount} euros, is far above what the contract says.",
        "My loan application was rejected after {day} days and the {amount} euros I paid for the appraisal were not refunded."],
    "Mortgage": [
        "The interest rate applied to my mortgage does not match the offer I signed, and I have overpaid {amount} euros.",
        "An insurance premium of {amount} euros was added to my mortgage instalment without my consent.",
        "My request to renegotiate the mortgage has been pending for weeks and the delay has cost me {amount} euros."],
    "Investments": [
        "My advisor moved my savings into a fund I never approved and the position is now {amount} euros below what I invested.",
        "The costs of the investment product were never explained to me and I have paid {amount} euros in charges.",
        "I asked to sell my fund units on the {day}th; the order was executed days later at a loss of {amount} euros."],
    "Insurance": [
        "My claim of {amount} euros on the policy linked to my account was rejected without any explanation.",
        "The insurance policy was renewed and {amount} euros charged although I had cancelled it in writing."],
}
spike_opening = [
    "Since this morning the app shows the same payment of {amount} euros twice and I cannot get through to anyone.",
    "After the update the app would not let me log in, and a payment of {amount} euros was taken twice.",
    "The mobile app has been down for two days and my payment of {amount} euros bounced."]
tone = {
    "very negative": ["This is unacceptable and I am furious.", "I have never been treated so badly by a bank.",
                      "I demand an immediate refund and a written apology.", "If this is not resolved within a week I will take it to the ombudsman."],
    "negative": ["I am very disappointed with how this has been handled.", "This has caused me a lot of stress and wasted time.",
                 "I expect a refund and a clear explanation."],
    "neutral": ["I would like this to be checked and corrected.", "Please let me know what happened and how it will be fixed.",
                "I assume it is a mistake, but it needs to be resolved."],
    "positive": ["I have always been happy with the bank, so I am sure this can be sorted out quickly.",
                 "Your staff have been helpful so far; I just need this last point solved.", "Thank you in advance for looking into it."],
}
contacts_sentence = ["I have already called {n} times and nobody has called me back.",
                     "This is my {nth} attempt to get an answer from you.",
                     "After {n} contacts with the branch and the call centre, nothing has changed."]

# the true amount: before the MNAR blank, and before the x100 keying error
x100_ids = [c for c, a in answer_key if a.startswith("univariate outlier: disputed_amount x100")]
true_amount = df["disputed_amount"].fillna(amount_before_mnar)
true_amount[df["complaint_id"].isin(x100_ids)] = true_amount[df["complaint_id"].isin(x100_ids)] / 100

is_spike = ~is_base & df["complaint_date"].between("2026-02-03", "2026-02-16")
sentiment_filled = df["sentiment_score"].fillna(0)
bucket = np.select([sentiment_filled < -0.55, sentiment_filled < -0.2, sentiment_filled < 0.2],
                   ["very negative", "negative", "neutral"], default="positive")

texts = []
for i in range(len(df)):
    amount = f"{true_amount.iloc[i]:,.0f}"
    day = rng.choice([4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 24, 25, 26, 27])
    pool = spike_opening if is_spike.iloc[i] else opening[df["product"].iloc[i]]
    parts = [rng.choice(pool).format(amount=amount, day=day)]
    n_contacts = int(df["contacts_before_complaint"].iloc[i])
    if n_contacts >= 3:
        nth = {3: "third", 4: "fourth", 5: "fifth", 6: "sixth", 7: "seventh", 8: "eighth", 9: "ninth"}.get(n_contacts, f"{n_contacts}th")
        parts.append(rng.choice(contacts_sentence).format(n=n_contacts, nth=nth))
    if df["prior_complaints_12m"].iloc[i] >= 2:
        parts.append("This is not the first time I have had to complain about this.")
    if df["days_since_event"].iloc[i] > 45:
        parts.append("The problem dates back more than a month and is still open.")
    parts.append(rng.choice(tone[bucket[i]]))
    texts.append(" ".join(parts))

df.insert(df.columns.get_loc("sentiment_score") + 1, "complaint_excerpt", texts)
df.loc[df["sentiment_score"].isna(), "complaint_excerpt"] = np.nan      # no written text, no excerpt
print("Excerpts written:", df["complaint_excerpt"].notna().sum(), " words per excerpt (median):",
      int(df["complaint_excerpt"].dropna().str.split().str.len().median()))
df.loc[df["complaint_excerpt"].notna(), ["complaint_id", "product", "sentiment_score", "complaint_excerpt"]].head(5)

## 12. Messy layer, part 4: duplicated rows and inconsistent labels

- six rows are duplicated (same `complaint_id`, same values), as happens when an export is run twice;
- twelve rows use inconsistent spellings of the channel (`E-mail`, `email`, `Phone ` with a trailing space).

In [ ]:
i = pick_rows(12)
df.loc[i, "channel"] = ["E-mail"] * 5 + ["email"] * 4 + ["Phone "] * 3
answer_key += [(c, "inconsistent channel label") for c in df.loc[i, "complaint_id"]]

dup = pick_rows(6)
answer_key += [(c, "duplicated row") for c in df.loc[dup, "complaint_id"]]
df = pd.concat([df, df.loc[dup]], ignore_index=True)      # done last: from here on the row count changes

df = df.sort_values(["complaint_date", "complaint_id"], kind="stable").reset_index(drop=True)
print(df.shape)

## 13. Verification: do the injected anomalies behave as intended?

We reproduce the checks the demo will run and make sure that:

- the multivariate outliers exceed the 99.9% Mahalanobis threshold while staying invisible to the univariate
  z-score (|z| < 3) and to every pairwise scatter plot;
- the univariate outliers are flagged by |z| > 3 or by the 1.5 × IQR rule;
- only a handful of "natural" rows exceed the Mahalanobis threshold, so the injected ones stand out.

In [ ]:
key_df = pd.DataFrame(answer_key, columns=["complaint_id", "anomaly"])
check = df.drop_duplicates()
errors = key_df.loc[key_df["anomaly"].str.startswith(("univariate", "bivariate")), "complaint_id"]
check = check[~check["complaint_id"].isin(errors)]                     # the demo removes gross errors before Mahalanobis

Xc = check[numeric_cols].copy()
Xc["disputed_amount"] = np.log1p(Xc["disputed_amount"])
Xc = Xc.dropna()
mu_c, S_c = Xc.mean().values, np.cov(Xc.values, rowvar=False)
diff = Xc.values - mu_c
d2 = np.einsum("ij,jk,ik->i", diff, np.linalg.inv(S_c), diff)
z = np.abs(diff / Xc.std().values)
pair_d2 = np.max([np.einsum("ij,jk,ik->i", diff[:, p], np.linalg.inv(S_c[np.ix_(p, p)]), diff[:, p]) for p in pairs], axis=0)

report = pd.DataFrame({"complaint_id": check.loc[Xc.index, "complaint_id"].values,
                       "mahalanobis_d2": d2, "max_abs_z": z.max(axis=1), "max_pair_d2": pair_d2})
report["rank"] = report["mahalanobis_d2"].rank(ascending=False).astype(int)
multi_ids = key_df.loc[key_df["anomaly"].str.startswith("multivariate"), "complaint_id"]
print(f"threshold (99.9%, 8 features) = {threshold_8d:.1f}; rows above threshold = {(d2 > threshold_8d).sum()}")
report[report["complaint_id"].isin(multi_ids)].round(2)

In [ ]:
# univariate check: is each injected univariate outlier caught by |z| > 3 (or by the 1.5 x IQR rule)?
uni_ids = key_df.loc[key_df["anomaly"].str.startswith("univariate"), "complaint_id"]
rows = df[df["complaint_id"].isin(uni_ids)].drop_duplicates("complaint_id")
for col in ["disputed_amount", "customer_age", "text_length_words", "days_since_event"]:
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75]); upper = q3 + 1.5 * (q3 - q1)
    zc = (rows[col] - s.mean()) / s.std()
    caught = rows.loc[(zc.abs() > 3) | (rows[col] > upper), "complaint_id"].tolist()
    print(f"{col:20s} flagged: {caught}")

## 14. Save the files

In [ ]:
df.to_csv("bank_complaints.csv", index=False)
key_df.sort_values(["anomaly", "complaint_id"]).to_csv("bank_complaints_answer_key.csv", index=False)

print("bank_complaints.csv:", df.shape)
print("\nMissing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nInjected anomalies:")
print(key_df["anomaly"].value_counts())

## 15. Labelled evaluation set for the prompt-engineering demo

The short prompt-engineering demo of Module 4 needs a few complaints with "expert" labels to score a prompt
against. We derive them from what the generator knows: the **topic** follows from the opening sentence of the
excerpt (a duplicated card charge is a fee dispute, an unknown transaction is fraud, a blocked account is service
quality, any mortgage or investment issue takes its product as topic), and the **tone level** follows from the
sentiment bucket used to write the text (0 calm or positive, 1 neutral, 2 annoyed, 3 angry or threatening
escalation).

The file holds 45 written complaints: 5 marked `example`, to be used as few-shot examples inside the prompt,
and 40 marked `test`, on which the prompts are scored. A separate random generator is used, so the main
dataset is not affected. In a real project these labels would come from complaint handlers, with the same rule
book that the prompt will use — that is the point the demo makes.

In [ ]:
topic_of_opening = {}
for product_name, sentences in opening.items():
    topics = {"Current account": ["Fraud", "Service quality", "Fees", "Service quality"],
              "Credit card": ["Fees", "Fraud", "Service quality", "Fees"],
              "Personal loan": ["Fees", "Fees", "Fees"],
              "Mortgage": ["Mortgage"] * 3, "Investments": ["Investments"] * 3,
              "Insurance": ["Other", "Fees"]}[product_name]
    for sentence, topic in zip(sentences, topics):
        topic_of_opening[sentence.split("{")[0]] = topic          # the fixed part before the first placeholder
for sentence in spike_opening:
    topic_of_opening[sentence.split("{")[0]] = "Service quality"

written = df.drop_duplicates("complaint_id")
written = written[written["complaint_excerpt"].notna()].copy()
written["topic"] = [next(t for prefix, t in topic_of_opening.items() if text.startswith(prefix)) for text in written["complaint_excerpt"]]
written["tone_level"] = np.select([written["sentiment_score"] >= 0.2, written["sentiment_score"] >= -0.2, written["sentiment_score"] >= -0.55], [0, 1, 2], default=3)

rng_eval = np.random.default_rng(99)                              # independent of the main generator
test_rows = pd.concat([written[written["tone_level"] == level].sample(10, random_state=int(rng_eval.integers(1e6))) for level in range(4)])
remaining = written.drop(test_rows.index)
example_rows = pd.concat([remaining[remaining["tone_level"] == level].sample(1, random_state=int(rng_eval.integers(1e6))) for level in [3, 2, 1, 0]]
                         + [remaining[remaining["topic"] == "Fraud"].sample(1, random_state=int(rng_eval.integers(1e6)))])

eval_set = pd.concat([example_rows.assign(split="example"), test_rows.assign(split="test")])
eval_set = eval_set[["complaint_id", "split", "product", "complaint_excerpt", "topic", "tone_level"]]
eval_set.to_csv("complaint_eval_set.csv", index=False)
print("complaint_eval_set.csv:", eval_set.shape)
print(eval_set["topic"].value_counts().to_dict())
print(eval_set["tone_level"].value_counts().sort_index().to_dict())